In [1]:
# ================================
# TRAINING + PREPROCESSING SCRIPT
# ================================

import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# =====================================
# 1. LOAD RAW DATA
# =====================================
print("Loading dataset...")

df = pd.read_csv("employee_reviews.csv")   # ← your dataset here
print("Original Shape:", df.shape)

# =====================================
# 2. DATA CLEANING (DETAILED)
# =====================================

# ---- STEP 1: Standardize column names ----
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Expected: company, location, job_title, work_balance_score, culture_values_score, compensation_benefit_score, career_opportunities_score, senior_management_score, overall_rating

# ---- STEP 2: Remove duplicate rows ----
df.drop_duplicates(inplace=True)

# ---- STEP 3: Replace "none"/"NaN" strings with proper NaN ----
df.replace(["none", "None", "N/A", "na", ""], np.nan, inplace=True)

# ---- STEP 4: Convert appropriate columns to numeric ----
numeric_cols = [
    "work_balance_score",
    "culture_values_score",
    "compensation_benefit_score",
    "career_opportunities_score",
    "senior_management_score",
    "overall_rating"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ---- STEP 5: Handle missing numeric values with median ----
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# ---- STEP 6: Clean text features ----
df["company"] = df["company"].astype(str).str.strip()
df["location"] = df["location"].astype(str).str.strip()
df["job_title"] = df["job_title"].astype(str).str.strip()

# ---- STEP 7: Remove rows with missing company names (critical field) ----
df = df[df["company"].notna()]

print("After cleaning shape:", df.shape)

# =====================================
# 3. AGGREGATE TO COMPANY-LEVEL SUMMARY
# =====================================

print("Aggregating company-level scores...")

company_df = df.groupby("company").agg({
    "work_balance_score": "mean",
    "culture_values_score": "mean",
    "compensation_benefit_score": "mean",
    "career_opportunities_score": "mean",
    "senior_management_score": "mean",
    "overall_rating": "mean"
}).reset_index()

company_df.to_csv("cleaned_data.csv", index=False)
print("Saved cleaned_data.csv")

# =====================================
# 4. DEFINE ML INPUT + TARGET
# =====================================

X = company_df.drop(columns=["overall_rating"])
y = company_df["overall_rating"]

categorical_cols = ["company"]
numeric_cols_for_model = [
    "work_balance_score",
    "culture_values_score",
    "compensation_benefit_score",
    "career_opportunities_score",
    "senior_management_score"
]

# =====================================
# 5. ENCODING + SCALING PIPELINE
# =====================================

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", MinMaxScaler(), numeric_cols_for_model)
    ]
)

# =====================================
# 6. XGBOOST MODEL
# =====================================

model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.08,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])

# =====================================
# 7. SPLIT TRAIN–TEST & TRAIN MODEL
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training model...")
pipeline.fit(X_train, y_train)

# =====================================
# 8. MODEL EVALUATION
# =====================================

score = pipeline.score(X_test, y_test)
print("Model R² Score:", round(score, 4))

# =====================================
# 9. SAVE MODEL + FEATURE MAPPING
# =====================================

joblib.dump(pipeline, "model.pkl")
print("Saved model.pkl")

# Save feature order for Flask
feature_order = {
    "categorical": categorical_cols,
    "numeric": numeric_cols_for_model
}

with open("feature_order.json", "w") as f:
    json.dump(feature_order, f, indent=4)

print("Saved feature_order.json")

print("\n============================")
print("TRAINING + CLEANING COMPLETE")
print("============================")


ModuleNotFoundError: No module named 'xgboost'